# ANATOMY ↔ GENE Relation-Wise Merge

Merges ANATOMY–GENE triples from multiple KG sources (DRKG, PrimeKG, Hetionet, TARKG),
aligns to a common schema, deduplicates by `(head, relation, tail)`, and saves the result.

## 0. Configuration

In [23]:
import pandas as pd
import numpy as np

# ── Base directories ──────────────────────────────────────────────────────────
BASE_DIR  = '/storage/Arushi/090526_EvoAge/kg_formation/'
PROC_DIR  = BASE_DIR + 'processed_data/'

# ── Output path ───────────────────────────────────────────────────────────────
OUT_PATH  = BASE_DIR + 'processed_data_relation_wise_merge/generalised/ANATOMY_GENE/ALL_ANATOMY_GENE.csv'

# ── Required output schema ────────────────────────────────────────────────────
REQUIRED_COLS = [
    'head', 'relation', 'tail',
    'head_type', 'relation_type', 'tail_type',
    'kg_source',
    'head_id_is', 'tail_id_is',
    'head_detail_name', 'tail_detail_name', 'kg_type', 'species'
]

## 1. Build Gene Lookup Dictionaries

Load NCBI gene info and ENSEMBL→NCBI mapping to create:
- `NCBI_ALL_Symb_Desc_dict`  — Symbol → description
- `exploded_dict_NCBI_ALL_GENE_Syn_Dict` — synonym → canonical Symbol

In [24]:
# ENSEMBL ID → NCBI Symbol mapping
ENS_2NCBI = pd.read_csv(f'{BASE_DIR}data_collection/databases_for_mapping/ENSEMBL/ENSEMBLE_ID_2_NCBI_Gene_SYM.csv')
ENS_2NCBI = ENS_2NCBI[~ENS_2NCBI['name'].isna()]
NCBI_2ENS__dict = dict(zip(ENS_2NCBI['name'], ENS_2NCBI['initial_alias']))
del ENS_2NCBI

# Full NCBI gene table (multi-species)
NCBI_ALL_GENE = pd.read_csv(f'{BASE_DIR}data_collection/databases_for_mapping/ncbi/Homo_sapiens.gene_info', sep='\t')
NCBI_ALL_GENE['ENSEMBLE_ID'] = NCBI_ALL_GENE['Symbol'].map(NCBI_2ENS__dict)

# Lookup dicts
NCBI_ALL_Symb_Desc_dict   = dict(zip(NCBI_ALL_GENE['Symbol'],  NCBI_ALL_GENE['description']))
NCBI_ALL_GENEname_dict    = dict(zip(NCBI_ALL_GENE['GeneID'],  NCBI_ALL_GENE['description']))
NCBI_ALL_GENEIDname_dict  = dict(zip(NCBI_ALL_GENE['GeneID'],  NCBI_ALL_GENE['Symbol']))

# Exploded synonym → Symbol dict
NCBI_ALL_GENE_Syn_Dict = dict(zip(NCBI_ALL_GENE['Synonyms'], NCBI_ALL_GENE['Symbol']))
exploded_dict_NCBI_ALL_GENE_Syn_Dict = {}
for k, v in NCBI_ALL_GENE_Syn_Dict.items():
    for alias in k.split('|'):
        exploded_dict_NCBI_ALL_GENE_Syn_Dict[alias.strip()] = v

print(f"NCBI gene table: {len(NCBI_ALL_GENE):,} rows")
print(f"Synonym dict:    {len(exploded_dict_NCBI_ALL_GENE_Syn_Dict):,} entries")

NCBI gene table: 193,505 rows
Synonym dict:    69,564 entries


In [25]:
UBERON = pd.read_csv(f'{BASE_DIR}data_collection/databases_for_mapping/uberon/Uberon_ID_Name_non_dup.csv')
UBERON_dict = dict(zip(UBERON['UBERON ID'], UBERON['Name']))
# UBERON_dict

## 2. Load Individual KG Sources

### 2a. DRKG

In [26]:
DRKG_Anatomy_GENE = pd.read_csv(f'{BASE_DIR}processed_data/DRKG/DRKG_Anatomy_Gene.csv')
DRKG_Anatomy_GENE.columns = DRKG_Anatomy_GENE.columns.str.lower()
DRKG_Anatomy_GENE['head_detail_name'] = DRKG_Anatomy_GENE['head'].map(UBERON_dict)
print(f"DRKG:     {len(DRKG_Anatomy_GENE):,} rows | columns: {list(DRKG_Anatomy_GENE.columns)}")
DRKG_Anatomy_GENE['kg_type'] = 'Generalised'
DRKG_Anatomy_GENE['species'] = 'Homo sapiens'
DRKG_Anatomy_GENE

DRKG:     726,156 rows | columns: ['head', 'relation', 'tail', 'head_type', 'relation_type', 'tail_type', 'kg_source', 'tail_detail_name', 'tail_ens', 'head_id_is', 'tail_id_is', 'head_orignal', 'tail_orignal', 'head_detail_name']


,head,relation,tail,head_type,relation_type,tail_type,kg_source,tail_detail_name,tail_ens,head_id_is,tail_id_is,head_orignal,tail_orignal,head_detail_name,kg_type,species
0,UBERON:0002082,Anatomy_Gene,CHD2,Anatomy,Hetionet::AdG::Anatomy:Gene,Gene,DRKG,chromodomain helicase DNA binding protein 2,ENSG00000173575,UBERON_ID,NCBI_ID,Anatomy::UBERON:0002082,Gene::1106,cardiac ventricle,Generalised,Homo sapiens
1,UBERON:0002185,Anatomy_Gene,GRIA2,Anatomy,Hetionet::AdG::Anatomy:Gene,Gene,DRKG,glutamate ionotropic receptor AMPA type subunit 2,ENSG00000120251,UBERON_ID,NCBI_ID,Anatomy::UBERON:0002185,Gene::2891,bronchus,Generalised,Homo sapiens
2,UBERON:0002240,Anatomy_Gene,GLE1,Anatomy,Hetionet::AdG::Anatomy:Gene,Gene,DRKG,GLE1 RNA export mediator,ENSG00000119392,UBERON_ID,NCBI_ID,Anatomy::UBERON:0002240,Gene::2733,spinal cord,Generalised,Homo sapiens
3,UBERON:0000955,Anatomy_Gene,HRC,Anatomy,Hetionet::AdG::Anatomy:Gene,Gene,DRKG,histidine rich calcium binding protein,ENSG00000130528,UBERON_ID,NCBI_ID,Anatomy::UBERON:0000955,Gene::3270,brain,Generalised,Homo sapiens
4,UBERON:0000996,Anatomy_Gene,EFHD1,Anatomy,Hetionet::AdG::Anatomy:Gene,Gene,DRKG,EF-hand domain family member D1,ENSG00000115468,UBERON_ID,NCBI_ID,Anatomy::UBERON:0000996,Gene::80303,vagina,Generalised,Homo sapiens
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
726151,UBERON:0000057,Anatomy_Gene,NDRG4,Anatomy,Hetionet::AeG::Anatomy:Gene,Gene,DRKG,NDRG family member 4,ENSG00000103034,UBERON_ID,NCBI_ID,Anatomy::UBERON:0000057,Gene::65009,urethra,Generalised,Homo sapiens
726152,UBERON:0000474,Anatomy_Gene,CDK5RAP3,Anatomy,Hetionet::AeG::Anatomy:Gene,Gene,DRKG,CDK5 regulatory subunit associated protein 3,ENSG00000108465,UBERON_ID,NCBI_ID,Anatomy::UBERON:0000474,Gene::80279,female reproductive system,Generalised,Homo sapiens
726153,UBERON:0002048,Anatomy_Gene,CLTA,Anatomy,Hetionet::AeG::Anatomy:Gene,Gene,DRKG,clathrin light chain A,ENSG00000122705,UBERON_ID,NCBI_ID,Anatomy::UBERON:0002048,Gene::1211,lung,Generalised,Homo sapiens
726154,UBERON:0002048,Anatomy_Gene,HCAR3,Anatomy,Hetionet::AeG::Anatomy:Gene,Gene,DRKG,hydroxycarboxylic acid receptor 3,ENSG00000255398,UBERON_ID,NCBI_ID,Anatomy::UBERON:0002048,Gene::8843,lung,Generalised,Homo sapiens


In [27]:
DRKG_Anatomy_GENE[DRKG_Anatomy_GENE['head_detail_name'].isna()]

,head,relation,tail,head_type,relation_type,tail_type,kg_source,tail_detail_name,tail_ens,head_id_is,tail_id_is,head_orignal,tail_orignal,head_detail_name,kg_type,species


### 2b. PrimeKG

In [28]:
Prime_Anatomy_GENE = pd.read_csv(f'{BASE_DIR}processed_data/PrimeKG/PriemKG_Anatomy_Gene.csv')
Prime_Anatomy_GENE.columns = Prime_Anatomy_GENE.columns.str.lower()
Prime_Anatomy_GENE['tail_id_is'] = 'NCBI_ID'
print(f"PrimeKG:  {len(Prime_Anatomy_GENE):,} rows | columns: {list(Prime_Anatomy_GENE.columns)}")
Prime_Anatomy_GENE['kg_type'] = 'Generalised'
Prime_Anatomy_GENE['species'] = 'Homo sapiens'
Prime_Anatomy_GENE.head(2)

PrimeKG:  1,518,203 rows | columns: ['head', 'relation', 'tail', 'head_type', 'relation_type', 'tail_type', 'kg_source', 'head_id_is', 'tail_id_is', 'head_detail_name', 'tail_alias_mapped', 'tail_detail_name', 'tail_ens']


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_alias_mapped,tail_detail_name,tail_ens,kg_type,species
0,UBERON:0000002,Anatomy_Gene,TSPAN6,Anatomy,expression present,Gene,PrimeKG,UBERON_ID,NCBI_ID,uterine cervix,TSPAN6,tetraspanin 6,ENSG00000000003,Generalised,Homo sapiens
1,UBERON:0000006,Anatomy_Gene,TSPAN6,Anatomy,expression present,Gene,PrimeKG,UBERON_ID,NCBI_ID,islet of Langerhans,TSPAN6,tetraspanin 6,ENSG00000000003,Generalised,Homo sapiens


In [29]:
Prime_Anatomy_GENE[Prime_Anatomy_GENE['head_detail_name'].isna()]

,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_alias_mapped,tail_detail_name,tail_ens,kg_type,species


### 2c. Hetionet

In [30]:
Hetionet_Anatomy_GENE = pd.read_csv(f'{BASE_DIR}processed_data/Hetionet/Anatomy_Gene_Hetionet.csv')
Hetionet_Anatomy_GENE.columns = Hetionet_Anatomy_GENE.columns.str.lower()
Hetionet_Anatomy_GENE['head_id_is'] = 'UBERON_ID'
Hetionet_Anatomy_GENE = Hetionet_Anatomy_GENE.rename(columns={'head_name': 'head_detail_name'})
print(f"Hetionet: {len(Hetionet_Anatomy_GENE):,} rows | columns: {list(Hetionet_Anatomy_GENE.columns)}")
Hetionet_Anatomy_GENE['kg_type'] = 'Generalised'
Hetionet_Anatomy_GENE['species'] = 'Homo sapiens'
Hetionet_Anatomy_GENE

Hetionet: 726,156 rows | columns: ['head', 'relation', 'tail', 'head_type', 'relation_type', 'tail_type', 'kg_source', 'head_id_is', 'tail_id_is', 'head_detail_name', 'tail_name', 'tail_detail_name']


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_name,tail_detail_name,kg_type,species
0,UBERON:0002082,Anatomy_Gene,CHD2,Anatomy,Anatomy–downregulates–Gene,Gene,Hetionet,UBERON_ID,NCBI_ID,cardiac ventricle,1106,chromodomain helicase DNA binding protein 2,Generalised,Homo sapiens
1,UBERON:0002185,Anatomy_Gene,GRIA2,Anatomy,Anatomy–downregulates–Gene,Gene,Hetionet,UBERON_ID,NCBI_ID,bronchus,2891,glutamate ionotropic receptor AMPA type subunit 2,Generalised,Homo sapiens
2,UBERON:0002240,Anatomy_Gene,GLE1,Anatomy,Anatomy–downregulates–Gene,Gene,Hetionet,UBERON_ID,NCBI_ID,spinal cord,2733,GLE1 RNA export mediator,Generalised,Homo sapiens
3,UBERON:0000955,Anatomy_Gene,HRC,Anatomy,Anatomy–downregulates–Gene,Gene,Hetionet,UBERON_ID,NCBI_ID,brain,3270,histidine rich calcium binding protein,Generalised,Homo sapiens
4,UBERON:0000996,Anatomy_Gene,EFHD1,Anatomy,Anatomy–downregulates–Gene,Gene,Hetionet,UBERON_ID,NCBI_ID,vagina,80303,EF-hand domain family member D1,Generalised,Homo sapiens
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
726151,UBERON:0000057,Anatomy_Gene,NDRG4,Anatomy,Anatomy–expresses–Gene,Gene,Hetionet,UBERON_ID,NCBI_ID,urethra,65009,NDRG family member 4,Generalised,Homo sapiens
726152,UBERON:0000474,Anatomy_Gene,CDK5RAP3,Anatomy,Anatomy–expresses–Gene,Gene,Hetionet,UBERON_ID,NCBI_ID,female reproductive system,80279,CDK5 regulatory subunit associated protein 3,Generalised,Homo sapiens
726153,UBERON:0002048,Anatomy_Gene,CLTA,Anatomy,Anatomy–expresses–Gene,Gene,Hetionet,UBERON_ID,NCBI_ID,lung,1211,clathrin light chain A,Generalised,Homo sapiens
726154,UBERON:0002048,Anatomy_Gene,HCAR3,Anatomy,Anatomy–expresses–Gene,Gene,Hetionet,UBERON_ID,NCBI_ID,lung,8843,hydroxycarboxylic acid receptor 3,Generalised,Homo sapiens


In [31]:
Hetionet_Anatomy_GENE[Hetionet_Anatomy_GENE['head_detail_name'].isna()], Hetionet_Anatomy_GENE[Hetionet_Anatomy_GENE['tail_detail_name'].isna()] 

(Empty DataFrame
 Columns: [head, relation, tail, head_type, relation_type, tail_type, kg_source, head_id_is, tail_id_is, head_detail_name, tail_name, tail_detail_name, kg_type, species]
 Index: [],
 Empty DataFrame
 Columns: [head, relation, tail, head_type, relation_type, tail_type, kg_source, head_id_is, tail_id_is, head_detail_name, tail_name, tail_detail_name, kg_type, species]
 Index: [])

### 2d. TARKG

In [32]:
TARKG_Anatomy_GENE = pd.read_csv(f'{BASE_DIR}processed_data/TARKG/Anatomy_Gene.csv')
TARKG_Anatomy_GENE.columns = TARKG_Anatomy_GENE.columns.str.lower()
print(f"TARKG:    {len(TARKG_Anatomy_GENE):,} rows | columns: {list(TARKG_Anatomy_GENE.columns)}")
TARKG_Anatomy_GENE['kg_type'] = 'Generalised'
TARKG_Anatomy_GENE['species'] = 'Homo sapiens'
TARKG_Anatomy_GENE.head(2)

TARKG:    1,446,844 rows | columns: ['head', 'relation', 'tail', 'head_type', 'relation_type', 'tail_type', 'kg_source', 'head_detail_name', 'tail_detail_name', 'head_id_is', 'tail_id_is', 'kg_index', 'kg', 'change']


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_detail_name,tail_detail_name,head_id_is,tail_id_is,kg_index,kg,change,kg_type,species
0,UBERON:0002185,Anatomy_Gene,TOP2A,Anatomy,AeG,Gene,TARKG,bronchus,DNA topoisomerase II alpha,UBERON_ID,NCBI_ID,Hetionet-2130105,Hetionet,0,Generalised,Homo sapiens
1,UBERON:0002185,Anatomy_Gene,TOP2A,Anatomy,Hetionet::AeG::Anatomy:Gene,Gene,TARKG,bronchus,DNA topoisomerase II alpha,UBERON_ID,NCBI_ID,DRKG-4001307,DRKG,0,Generalised,Homo sapiens


In [33]:
TARKG_Anatomy_GENE[TARKG_Anatomy_GENE['head_detail_name'].isna()]

,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_detail_name,tail_detail_name,head_id_is,tail_id_is,kg_index,kg,change,kg_type,species


## 3. Align Schemas and Concatenate

In [34]:
source_dfs = [
    DRKG_Anatomy_GENE,
    Prime_Anatomy_GENE,
    Hetionet_Anatomy_GENE,
    TARKG_Anatomy_GENE,
]

aligned = []
for df in source_dfs:
    df = df.copy()
    for col in REQUIRED_COLS:
        if col not in df.columns:
            df[col] = None       
    aligned.append(df[REQUIRED_COLS])

final_df = pd.concat(aligned, ignore_index=True)
print(f"Consolidated rows: {len(final_df):,}")
final_df['species'] = 'Homo sapiens'
final_df

Consolidated rows: 4,417,359


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species
0,UBERON:0002082,Anatomy_Gene,CHD2,Anatomy,Hetionet::AdG::Anatomy:Gene,Gene,DRKG,UBERON_ID,NCBI_ID,cardiac ventricle,chromodomain helicase DNA binding protein 2,Generalised,Homo sapiens
1,UBERON:0002185,Anatomy_Gene,GRIA2,Anatomy,Hetionet::AdG::Anatomy:Gene,Gene,DRKG,UBERON_ID,NCBI_ID,bronchus,glutamate ionotropic receptor AMPA type subunit 2,Generalised,Homo sapiens
2,UBERON:0002240,Anatomy_Gene,GLE1,Anatomy,Hetionet::AdG::Anatomy:Gene,Gene,DRKG,UBERON_ID,NCBI_ID,spinal cord,GLE1 RNA export mediator,Generalised,Homo sapiens
3,UBERON:0000955,Anatomy_Gene,HRC,Anatomy,Hetionet::AdG::Anatomy:Gene,Gene,DRKG,UBERON_ID,NCBI_ID,brain,histidine rich calcium binding protein,Generalised,Homo sapiens
4,UBERON:0000996,Anatomy_Gene,EFHD1,Anatomy,Hetionet::AdG::Anatomy:Gene,Gene,DRKG,UBERON_ID,NCBI_ID,vagina,EF-hand domain family member D1,Generalised,Homo sapiens
...,...,...,...,...,...,...,...,...,...,...,...,...,...
4417354,UBERON:0002097,Anatomy_Gene,KRTAP20-2,Anatomy,Hetionet::AeG::Anatomy:Gene,Gene,TARKG,UBERON_ID,NCBI_ID,skin of body,keratin associated protein 20-2,Generalised,Homo sapiens
4417355,UBERON:0001003,Anatomy_Gene,KRTAP20-2,Anatomy,AeG,Gene,TARKG,UBERON_ID,NCBI_ID,skin epidermis,keratin associated protein 20-2,Generalised,Homo sapiens
4417356,UBERON:0001003,Anatomy_Gene,KRTAP20-2,Anatomy,Hetionet::AeG::Anatomy:Gene,Gene,TARKG,UBERON_ID,NCBI_ID,skin epidermis,keratin associated protein 20-2,Generalised,Homo sapiens
4417357,UBERON:0001037,Anatomy_Gene,KRTAP20-2,Anatomy,AeG,Gene,TARKG,UBERON_ID,NCBI_ID,strand of hair,keratin associated protein 20-2,Generalised,Homo sapiens


In [35]:
final_df['relation'] = 'AnatomicalEntity_Gene'
final_df['head_type'] = 'AnatomicalEntity'


## 4. QC — NaN Check Before Name Repair

In [36]:
def nan_summary(df):
    true_nan   = df.isna().sum()
    string_nan = df.apply(lambda c: c.astype(str).str.upper().eq('NAN').sum())
    return pd.DataFrame({
        'NaN_count':          true_nan,
        "'NAN'_string_count": string_nan,
        'Total_NaN_like':     true_nan + string_nan
    })

nan_summary(final_df)

,NaN_count,'NAN'_string_count,Total_NaN_like
head,0,0,0
relation,0,0,0
tail,0,0,0
head_type,0,0,0
relation_type,0,0,0
tail_type,0,0,0
kg_source,0,0,0
head_id_is,0,0,0
tail_id_is,0,0,0
head_detail_name,0,0,0


## 5. Repair Missing `tail_detail_name`

For rows where `tail_detail_name` is NaN:
1. Strip leading `-` from the tail symbol.
2. Try mapping via the synonym dictionary.
3. Fall back to `NCBI_ALL_Symb_Desc_dict` (Symbol → description).

In [37]:
mask = final_df['tail_detail_name'].isna()
print(f"Rows needing tail_detail_name repair: {mask.sum():,}")

# Step 1 – clean symbol (remove leading '-')
final_df.loc[mask, 'tail'] = final_df.loc[mask, 'tail'].str.replace('-', '', regex=False)

# Step 2 – map through synonym dict
final_df.loc[mask, 'tail'] = (
    final_df.loc[mask, 'tail']
    .map(exploded_dict_NCBI_ALL_GENE_Syn_Dict)
    .fillna(final_df.loc[mask, 'tail'])
)

# Step 3 – re-check mask and fill description
mask2 = final_df['tail_detail_name'].isna()
final_df.loc[mask2, 'tail_detail_name'] = final_df.loc[mask2, 'tail'].map(NCBI_ALL_Symb_Desc_dict)

# Drop rows still missing tail_detail_name
before = len(final_df)
final_df = final_df[~final_df['tail_detail_name'].isna()].reset_index(drop=True)
print(f"Dropped {before - len(final_df):,} unresolvable rows → {len(final_df):,} remaining")

Rows needing tail_detail_name repair: 1,560
Dropped 66 unresolvable rows → 4,417,293 remaining


In [38]:
mask = final_df['tail_detail_name'].isna()
print(f"Rows needing tail_detail_name repair: {mask.sum():,}")

# Step 1 – clean symbol (remove leading '-')
final_df.loc[mask, 'tail'] = final_df.loc[mask, 'tail'].str.replace('-', '', regex=False)

# Step 2 – map through synonym dict
final_df.loc[mask, 'tail'] = (
    final_df.loc[mask, 'tail']
    .map(exploded_dict_NCBI_ALL_GENE_Syn_Dict)
    .fillna(final_df.loc[mask, 'tail'])
)

# Step 3 – re-check mask and fill description
mask2 = final_df['tail_detail_name'].isna()
final_df.loc[mask2, 'tail_detail_name'] = final_df.loc[mask2, 'tail'].map(NCBI_ALL_Symb_Desc_dict)

# Drop rows still missing tail_detail_name
before = len(final_df)
final_df = final_df[~final_df['tail_detail_name'].isna()].reset_index(drop=True)
print(f"Dropped {before - len(final_df):,} unresolvable rows → {len(final_df):,} remaining")

Rows needing tail_detail_name repair: 0
Dropped 0 unresolvable rows → 4,417,293 remaining


## 6. Deduplicate and Add Schema Columns

Group by `(head, relation, tail)` and collapse duplicate `kg_source` values with `::` separator.

In [39]:
GROUP_COLS = ['head', 'relation', 'tail']

def merge_sources(x):
    return '::'.join(sorted(set(x.dropna())))

final_df_dedup = final_df.groupby(GROUP_COLS, as_index=False).agg({
    'head_type':        'first',
    'relation_type':    'first',
    'tail_type':        'first',
    'kg_source':        merge_sources,
    'kg_type':          merge_sources,   # ← changed from 'first'
    'head_id_is':       'first',
    'tail_id_is':       'first',
    'head_detail_name': 'first',
    'tail_detail_name': 'first',
    'species': 'first'
})


# Enforce final column order
final_df_dedup = final_df_dedup[REQUIRED_COLS]

print(f"After deduplication: {len(final_df_dedup):,} rows")
final_df_dedup.head()

After deduplication: 1,721,930 rows


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species
0,UBERON:0000002,AnatomicalEntity_Gene,A1BG-AS1,AnatomicalEntity,expression present,Gene,PrimeKG,UBERON_ID,NCBI_ID,uterine cervix,A1BG antisense RNA 1,Generalised,Homo sapiens
1,UBERON:0000002,AnatomicalEntity_Gene,A2M,AnatomicalEntity,Hetionet::AuG::Anatomy:Gene,Gene,DRKG::Hetionet::TARKG,UBERON_ID,NCBI_ID,uterine cervix,alpha-2-macroglobulin,Generalised,Homo sapiens
2,UBERON:0000002,AnatomicalEntity_Gene,A2ML1,AnatomicalEntity,Hetionet::AuG::Anatomy:Gene,Gene,DRKG::Hetionet::PrimeKG::TARKG,UBERON_ID,NCBI_ID,uterine cervix,alpha-2-macroglobulin like 1,Generalised,Homo sapiens
3,UBERON:0000002,AnatomicalEntity_Gene,AAAS,AnatomicalEntity,Hetionet::AeG::Anatomy:Gene,Gene,DRKG::Hetionet::PrimeKG::TARKG,UBERON_ID,NCBI_ID,uterine cervix,aladin WD repeat nucleoporin,Generalised,Homo sapiens
4,UBERON:0000002,AnatomicalEntity_Gene,AACS,AnatomicalEntity,Hetionet::AeG::Anatomy:Gene,Gene,DRKG::Hetionet::PrimeKG::TARKG,UBERON_ID,NCBI_ID,uterine cervix,acetoacetyl-CoA synthetase,Generalised,Homo sapiens


In [40]:
print("\nkg_source values present:", set(final_df_dedup['kg_source']))


kg_source values present: {'DRKG::Hetionet::PrimeKG', 'DRKG::Hetionet', 'DRKG::Hetionet::TARKG', 'DRKG::Hetionet::PrimeKG::TARKG', 'PrimeKG'}


In [41]:
print("\nkg_type values present:", set(final_df_dedup['kg_type']))


kg_type values present: {'Generalised'}


## 7. QC — NaN Check After Deduplication

In [42]:
nan_summary(final_df_dedup)

,NaN_count,'NAN'_string_count,Total_NaN_like
head,0,0,0
relation,0,0,0
tail,0,0,0
head_type,0,0,0
relation_type,0,0,0
tail_type,0,0,0
kg_source,0,0,0
head_id_is,0,0,0
tail_id_is,0,0,0
head_detail_name,0,0,0


In [43]:
final_df_dedup['head_species'] = 'Homo sapiens'
final_df_dedup['tail_species'] = 'Homo sapiens'


## 8. Save Output

In [44]:
final_df_dedup.to_csv(OUT_PATH, index=False)
print(f"Saved {len(final_df_dedup):,} rows → {OUT_PATH}")

Saved 1,721,930 rows → /storage/Arushi/090526_EvoAge/kg_formation/processed_data_relation_wise_merge/generalised/ANATOMY_GENE/ALL_ANATOMY_GENE.csv


In [45]:
final_df_dedup

,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species,head_species,tail_species
0,UBERON:0000002,AnatomicalEntity_Gene,A1BG-AS1,AnatomicalEntity,expression present,Gene,PrimeKG,UBERON_ID,NCBI_ID,uterine cervix,A1BG antisense RNA 1,Generalised,Homo sapiens,Homo sapiens,Homo sapiens
1,UBERON:0000002,AnatomicalEntity_Gene,A2M,AnatomicalEntity,Hetionet::AuG::Anatomy:Gene,Gene,DRKG::Hetionet::TARKG,UBERON_ID,NCBI_ID,uterine cervix,alpha-2-macroglobulin,Generalised,Homo sapiens,Homo sapiens,Homo sapiens
2,UBERON:0000002,AnatomicalEntity_Gene,A2ML1,AnatomicalEntity,Hetionet::AuG::Anatomy:Gene,Gene,DRKG::Hetionet::PrimeKG::TARKG,UBERON_ID,NCBI_ID,uterine cervix,alpha-2-macroglobulin like 1,Generalised,Homo sapiens,Homo sapiens,Homo sapiens
3,UBERON:0000002,AnatomicalEntity_Gene,AAAS,AnatomicalEntity,Hetionet::AeG::Anatomy:Gene,Gene,DRKG::Hetionet::PrimeKG::TARKG,UBERON_ID,NCBI_ID,uterine cervix,aladin WD repeat nucleoporin,Generalised,Homo sapiens,Homo sapiens,Homo sapiens
4,UBERON:0000002,AnatomicalEntity_Gene,AACS,AnatomicalEntity,Hetionet::AeG::Anatomy:Gene,Gene,DRKG::Hetionet::PrimeKG::TARKG,UBERON_ID,NCBI_ID,uterine cervix,acetoacetyl-CoA synthetase,Generalised,Homo sapiens,Homo sapiens,Homo sapiens
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1721925,UBERON:0016525,AnatomicalEntity_Gene,ZBED4,AnatomicalEntity,expression present,Gene,PrimeKG,UBERON_ID,NCBI_ID,frontal lobe,zinc finger BED-type containing 4,Generalised,Homo sapiens,Homo sapiens,Homo sapiens
1721926,UBERON:0016525,AnatomicalEntity_Gene,ZC3H7A,AnatomicalEntity,expression present,Gene,PrimeKG,UBERON_ID,NCBI_ID,frontal lobe,zinc finger CCCH-type containing 7A,Generalised,Homo sapiens,Homo sapiens,Homo sapiens
1721927,UBERON:0016525,AnatomicalEntity_Gene,ZCRB1,AnatomicalEntity,expression present,Gene,PrimeKG,UBERON_ID,NCBI_ID,frontal lobe,zinc finger CCHC-type and RNA binding motif co...,Generalised,Homo sapiens,Homo sapiens,Homo sapiens
1721928,UBERON:0016525,AnatomicalEntity_Gene,ZNF143,AnatomicalEntity,expression present,Gene,PrimeKG,UBERON_ID,NCBI_ID,frontal lobe,zinc finger protein 143,Generalised,Homo sapiens,Homo sapiens,Homo sapiens
